In [ ]:
import json
with open("conversations_20251212_161733.json", encoding="utf-8") as f:
    data = json.load(f)

sess = set()
for k, ex in data.items():
    if len(ex["all_messages"]):
        if "2025-12-08" in ex["all_messages"][0]["timestamp"] and len(ex["puzzles"]) > 8:
            sess.update([k])

print(list(sess), len(sess))

In [ ]:
import csv
human_helper = []
human_worker = []
with open("collaboration_metrics_20251212_161733.csv", encoding="utf-8") as f:
    metrics = csv.DictReader(f)
    for m in metrics:
        if m["session_id"] not in sess:
        # ['a20f4c15-e3dc-4711-afd6-fa57321290ba', '60b53f2b-53de-47c8-878a-663540a6342f', 
        #                        'fc11c7d3-dda6-49b8-b0be-bd2ca9895964', 'e4963a82-d87e-48e1-9ac0-6a9998b854a7', 
        #                        'c0bea389-ca60-45ab-80f6-6625b12f8462', '3e048e72-d2d3-46c3-924d-1315df19869c', 
        #                        '51d12da4-b213-44c3-a2e5-e4fe0d4e894d', 'e80580db-31ae-441b-aa31-6a5ea9ebc2d6', 
        #                        '7686bcdb-b67e-4409-a997-14eafd823dc0', '50b77531-740c-4bfa-80d7-420ce74b8570']:
            continue
        if m["helper_mode"] == "human" and int(m["total_messages"]) > 100:
            human_helper.append(m["session_id"])
        if m["helper_mode"] == "llm" and int(m["total_messages"]) > 100:
            human_worker.append(m["session_id"])

In [ ]:
print(len(human_helper), len(human_worker))
print(human_helper)

In [ ]:
# # human_helper = ["a20f4c15-e3dc-4711-afd6-fa57321290ba"]
# ex = data["a20f4c15-e3dc-4711-afd6-fa57321290ba"]
# data[human_helper[2]]["puzzles"]["0"][0]
# m["all_messages"][0]
# human_helper
with open("sessions_12_08_2025.json", "w", encoding="utf-8") as f:
    json.dump({session_id: data[session_id] for session_id in human_helper + human_worker}, f)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# import numpy as np

# for session_id in human_helper:
#     d = data[session_id]["puzzles"]
#     print(session_id)
#     l1 = []
#     l2 = []
#     p_idx = []
#     i = 0
#     for p, ex in d.items():
#         if p == "null":
#             # print(p, len(ex))
#             continue
#         last = "none"
#         worker = []
#         helper = []
#         for i, e in enumerate([e_tmp for e_tmp in ex["messages"] if e_tmp["message"]]):
#             if i > 0 and "The board is empty" in e["message"]:
#                     continue
#             if last != e["sender"]:
#                 last = e["sender"]
#                 message = e["message"]
#             else:
#                 message = message + "\n" + e["message"]
#             if e["sender"] == "worker":
#                 # l1.append(len(message))
#                 worker.append(len(message))
#                 # p_idx.append(e["puzzle_index"])
#             if e["sender"] == "helper":
#                 helper.append(len(message))
#                 # l2.append(len(message))

#         l1.append(np.sum(worker))
#         l2.append(np.sum(helper))
#         p_idx.append(p)
    
#     end = min(len(l1), len(l2))
#     df = pd.DataFrame.from_dict({"human helper": l2[:end], "AI worker": l1[:end], "puzzle": p_idx[:end]})
    
#     ax = df[['human helper', 'AI worker']].plot()  
#     ax.set_xlabel('Turns')  
#     ax.set_ylabel('Characters (length)')  
      
#     # Find boundaries where puzzle changes  
#     boundaries = df.index[df['puzzle'].shift() != df['puzzle']]  
      
#     for b in boundaries:  
#         ax.axvline(x=b, color='gray', linestyle='--', alpha=0.5)  
      
#     plt.show() 
#     # break
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def print_session(human_sessions, aggr="sum", role="human_helper"):
    l1 = []
    l2 = []
    p_idx = []
    sessions = []
    for session_id in human_sessions:
        d = data[session_id]["puzzles"]
        i = 0
        for p, ex in d.items():
            if p == "null":
                # print(p, len(ex))
                continue
            last = "none"
            worker = []
            helper = []
            for i, e in enumerate([e_tmp for e_tmp in ex["messages"] if e_tmp["message"]]):
                if i > 0 and "The board is empty" in e["message"]:
                        continue
                if last != e["sender"]:
                    last = e["sender"]
                    message = e["message"]
                else:
                    message = message + "\n" + e["message"]
                if e["sender"] == "worker":
                    worker.append(len(message))
                if e["sender"] == "helper":
                    helper.append(len(message))

            if aggr == "sum":
                l1.append(np.sum(worker))
                l2.append(np.sum(helper))
            elif aggr == "mean":
                l1.append(np.mean(worker))
                l2.append(np.mean(helper))
            else:
                l1.append(len(worker))
                l2.append(len(helper))
                
            p_idx.append(p)
            sessions.append(session_id)
        
    end = min(len(l1), len(l2))

    if role == "human_helper":
        df = pd.DataFrame.from_dict({"human helper": l2[:end], "AI worker": l1[:end], "puzzle": p_idx[:end], "session": sessions})
        g = df.groupby('puzzle')  
          
        mean = g[['human helper','AI worker']].mean()  
        std  = g[['human helper','AI worker']].std()  
    else:
        df = pd.DataFrame.from_dict({"human worker": l1[:end], "AI helper": l2[:end], "puzzle": p_idx[:end], "session": sessions})
        g = df.groupby('puzzle')  
      
        mean = g[['human worker','AI helper']].mean()  
        std  = g[['human worker','AI helper']].std()  
      
    ax = mean.plot()#yerr=std)  
    ax.set_xlabel('Puzzle')  
    ax.set_ylabel(f'Characters (avg {aggr})') 
    ax.axvspan(0, 1, color='yellow', alpha=0.15, label='shared view (practice)')  
    ax.axvspan(1, 4, color='green', alpha=0.15, label='shared view')  
      
    ax.axvspan(5, 8, color='red', alpha=0.15, label='non‑shared view')  
    # ax.legend()  
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
     
    for b in range(0, 9):  
        ax.axvline(x=b, color='gray', linestyle='--', alpha=0.5) 
    
    plt.savefig(f"{role}_{aggr}_messages_shared_view_first.png", dpi=300, bbox_inches="tight")  

# print_session(human_helper, aggr="mean", role="human_helper")
print_session(human_helper, aggr="sum", role="human_helper")

In [ ]:
# print_session(human_worker, aggr="mean", role="human_worker")
print_session(human_worker, aggr="sum", role="human_worker")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

l1 = []
l2 = []
p_idx = []
sessions = []
for i, session_id in enumerate(human_worker + human_helper):
    d = data[session_id]["puzzles"]
    # print(session_id)

    i = 0
    for p, ex in d.items():
        if p == "null":
            # print(p, len(ex))
            continue
        last = "none"
        worker = []
        helper = []
        for i, e in enumerate([e_tmp for e_tmp in ex["messages"] if e_tmp["message"]]):

            if i > 0 and "The board is empty" in e["message"]:
                    continue
            if last != e["sender"]:
                last = e["sender"]
                message = e["message"]
            else:
                message = message + "\n" + e["message"]
            if e["sender"] == "worker":
                # l1.append(len(message))
                worker.append(len(message))
                # p_idx.append(e["puzzle_index"])
            if e["sender"] == "helper":
                helper.append(len(message))
                # l2.append(len(message))
        if session_id in human_worker:
            l1.append(len(worker+helper))
            p_idx.append(p)
            sessions.append(session_id)
        else:
            l2.append(len(worker+helper))
            p_idx.append(p)
            sessions.append(session_id)
    
end = min(len(l1), len(l2))
df = pd.DataFrame.from_dict({"with human worker": l1[:end], "with human helper": l2[:end], "puzzle": p_idx[:end], "session": sessions[:end]})
g = df.groupby('puzzle')  
  
mean = g[['with human worker','with human helper']].mean()  
std  = g[['with human worker','with human helper']].std()  
  
ax = mean.plot()#yerr=std)  
ax.set_xlabel('Puzzle')  
ax.set_ylabel('Turns (avg number)') 

ax.axvspan(0, 1, color='yellow', alpha=0.15, label='shared view (practice)') 
ax.axvspan(1, 4, color='green', alpha=0.15, label='shared view')  

ax.axvspan(5, 8, color='red', alpha=0.15, label='non-shared view')  
# ax.legend()  
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
for b in range(0, 9):  
    ax.axvline(x=b, color='gray', linestyle='--', alpha=0.5) 

plt.savefig("session_lengths_shared_view_first.png", dpi=300, bbox_inches="tight")  

In [ ]:
print(len(l1), len(l2))

In [ ]:
from azure.identity import (
    AzureCliCredential,
    ChainedTokenCredential,
    DefaultAzureCredential,
    get_bearer_token_provider,
)
import requests
import base64 
def gpt_request_chat(messages: list, 
                model: str = "gpt-4o_2024-08-06", 
                max_tokens: int = 1500,
                api_version: str = "2025-01-01-preview") -> str:
  """Make a request to Azure OpenAI's GPT model."""

  try:
    base_url = "https://trapi.research.microsoft.com/"
    trapi_endpoint = "msrc/shared/"
    url = f'{base_url}{trapi_endpoint}openai/deployments/{model}/chat/completions?api-version={api_version}'
    azure_credential = ChainedTokenCredential(
        AzureCliCredential(),
        DefaultAzureCredential(),
    )
    token_provider = get_bearer_token_provider(azure_credential, "api://trapi/.default")
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token_provider()}"
    }
    data = {  
        "messages": messages
    }
    nextResponse = requests.post(url, headers=headers, json=data)
    return nextResponse.json()["choices"][-1]["message"]["content"]
  except Exception as e:
    print(e)
    # print(nextResponse)
    return f"GENERATION ERROR: {str(e)}"

def gpt_request(prompt: str, 
                model: str = "gpt-4o_2024-08-06", 
                max_tokens: int = 1500,
                api_version: str = "2025-01-01-preview") -> str:
  """Make a request to Azure OpenAI's GPT model."""

  try:
    base_url = "https://trapi.research.microsoft.com/"
    trapi_endpoint = "msrc/shared/"
    url = f'{base_url}{trapi_endpoint}openai/deployments/{model}/chat/completions?api-version={api_version}'
    azure_credential = ChainedTokenCredential(
        AzureCliCredential(),
        DefaultAzureCredential(),
    )
    token_provider = get_bearer_token_provider(azure_credential, "api://trapi/.default")
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token_provider()}"
    }
    data = {  
        "messages": [{"role": "user", "content": prompt}]
    }
    nextResponse = requests.post(url, headers=headers, json=data)
    return nextResponse.json()["choices"][-1]["message"]["content"]
  except Exception as e:
    print(e)
    # print(nextResponse)
    return f"GENERATION ERROR: {str(e)}"

import ast
def get_json(response, default={}):  
    """  
    Extract JSON content from a response string.  
    """  
    response = response.replace("\n", " ").replace("\t", " ").replace("\'", "'")
    b_start = response.find("{")
    start = response.find("```json")  
    if start == -1:  
        start = response.find("```")
        if start > b_start:
            end = response.rfind("}")
            # print(response[b_start:end+1], start, end)
            try:
                return ast.literal_eval(response[b_start:end+1])
            except Exception as e:
                return default
        
        if start == -1:  
            start = 0  
        else:  
            start += len("```")
    else:  
        start += len("```json")  
  
    end = response.rfind("```", start)  
    if end == -1:  
        end = len(response)  

    if not response[start:end].strip().endswith("}"):
        # print(response[start:end], start, end)
        try: 
            # print(response[start:end], start, end)
            return ast.literal_eval(response[start:end] + "\n}")
        except Exception as e:
            return default

    try:  
    # print(response[start:end], start, end)
        return ast.literal_eval(response[start:end])  
    except Exception as e:
        if len(default) > 0:
            key = list(default.keys())[0]
            start = response.find(f"\"{key}\": ") + len(f"\"{key}\": ") + 1
            if start == -1:
                return {key: response}
                
            end = start + response[start:].find("\"")
            return {key: response[start:end]}
            
        print(e)
        return default  

def gpt_request_text_and_image_upload(prompt: str,  
                                      image_path: str,  
                                      model: str = "gpt-4o_2024-08-06",  
                                      max_tokens: int = 1500,  
                                      api_version: str = "2025-01-01-preview") -> str:  
    """  
    Make a request to Azure OpenAI's GPT model with both text and a locally uploaded image.  
    The image will be read from disk and sent as Base64 so the API can process it directly.  
    """  
    try:  
        # Read and Base64 encode the image  
        with open(image_path, "rb") as img_file:  
            image_base64 = base64.b64encode(img_file.read()).decode("utf-8")  
  
        base_url = "https://trapi.research.microsoft.com/"  
        trapi_endpoint = "msrc/shared/"  
        url = f'{base_url}{trapi_endpoint}openai/deployments/{model}/chat/completions?api-version={api_version}'  
        azure_credential = ChainedTokenCredential(  
            AzureCliCredential(),  
            DefaultAzureCredential(),  
        )  
        token_provider = get_bearer_token_provider(azure_credential, "api://trapi/.default")  
        headers = {  
            "Content-Type": "application/json",  
            "Authorization": f"Bearer {token_provider()}"  
        }  
  
        # Multimodal message with text + Base64 image  
        data = {  
            "messages": [  
                {  
                    "role": "user",  
                    "content": [  
                        {"type": "text", "text": prompt},
                        {"type": "image_url",
                        "image_url": {
                           "url": f"data:image/jpeg;base64,{image_base64}"
                        }
                        }
                    ]  
                }  
            ],  
            "max_tokens": max_tokens  
        }  
        response = requests.post(url, headers=headers, json=data)  
        response.raise_for_status()  
  
        return response.json()["choices"][-1]["message"]["content"]  
    except Exception as e:  
        print("Error:", e)  
        return f"GENERATION ERROR: {str(e)}"

In [ ]:
prompt1 = """
Analyze the following conversation and evaluate the pair’s Common Ground Formation. Focus only on shared vocabulary, referring expressions, and naming stability. Ignore task performance.

Conversation:
{conversation}

Your evaluation should include:

Lexicon Convergence:
Did the partners converge on consistent names or labels for objects?
Rate: High / Medium / Low

Reduction of Referring Expressions:
Did referring expressions become shorter, more efficient, or less descriptive over time?
Rate: High / Medium / Low

Stability of Labels:
Did the partners reuse the same labels without renegotiating them?
Count instances where labels changed or required renegotiation.
Rate: High / Medium / Low

Provide the results as JSON object:. 
```json
{
 "worker": {"Lexicon Convergence": "High / Medium / Low", "Reduction of Referring Expressions:": "High / Medium / Low", "Stability of Labels": "High / Medium / Low"},
 "helper": {"Lexicon Convergence": "High / Medium / Low", "Reduction of Referring Expressions:": "High / Medium / Low", "Stability of Labels": "High / Medium / Low"}
}
```
"""

prompt2 = """
Analyze the following conversation and evaluate the pair’s Situation Awareness. Focus only on awareness of task state, detection of errors, and the need for explicit alignment statements.

Conversation:
{conversation}

Your evaluation should include:

Monitoring Effectiveness:
Did the helper or partner consistently track the other’s actions?
Rate: High / Medium / Low

Intervention Timing:
How quickly did the partner intervene when a mistake occurred?
Rate: High / Medium / Low

Explicit Alignment Statements:
Count verbal alignments (e.g., "Ready?", "Done?", "Did you get it?").
More = weaker real-time situational awareness.
Rate: High / Medium / Low.

Provide the results as JSON object:. 
```json
{
 "worker": {"Monitoring Effectiveness": "High / Medium / Low", "Intervention Timing:": "High / Medium / Low", "Explicit Alignment Statement": "High / Medium / Low"},
 "helper": {"Monitoring Effectiveness": "High / Medium / Low", "Intervention Timing:": "High / Medium / Low", "Explicit Alignment Statement": "High / Medium / Low"}
}
```
"""

prompt3 = """
Analyze the following conversation and evaluate the Communication Process. Focus on how effectively the partners managed understanding using repairs, clarifications, acknowledgments, and spatial language. Do NOT evaluate grounding cost or effort.

Conversation:
{conversation}

Your evaluation should include:

Clarification Requests:
Count how many times they asked for clarification (e.g., “Which one?”, “Where?”).
Rate clarity problems: High / Medium / Low.

Repairs and Corrections:
Count self‑repairs and partner‑repairs (e.g., “No, not that one”).
Rate repair frequency: High / Medium / Low.

Acknowledgments:
Count explicit acknowledgments (“OK,” “Got it,” “Done”).
Count implicit ones (correct actions demonstrating understanding).
Rate: High / Medium / Low.

Spatial Language Use:
Count spatial deixis terms (“left,” “right,” “above,” etc.).
Evaluate whether spatial language was clear or ambiguous.
Rate: High / Medium / Low.

Provide the results as JSON object:. 
```json
{
 "worker": {"Clarification Requests": "High / Medium / Low", "Repairs and Corrections:": "High / Medium / Low", "Acknowledgments": "High / Medium / Low", "Spatial Language Use": "High / Medium / Low"},
 "helper": {"Clarification Requests": "High / Medium / Low", "Repairs and Corrections:": "High / Medium / Low", "Acknowledgments": "High / Medium / Low", "Spatial Language Use": "High / Medium / Low"}
}
```
"""


prompt4 = """
You judge how well participants establish mutual understanding about:

Puzzle pieces (object identity) and their locations (spatial placement)
Treat piece grounding and location grounding as the joint if they refer to
the same grounding targets, but return them as two annotations.

There are two roles: The helper does not see all available puzzle pieces but sees the target
pattern with the correct locations and pieces. The worker does not see the target puzzle but sees
all available pieces.

{view}

Input
You will receive:
A turn‑by‑turn transcript labeled by speaker (Helper / Worker) up to a current turn
The current turn to annotate

Core Objective
For each referential episode, evaluate whether the participants acts with the current turn 
in a grounding cycle consisting of:
Presentation
Clarification
Correction
Repair
Acceptance
Followup

Do this jointly for:
Piece identity
Location / spatial target

Grounding is successful when participants mutually accept understanding sufficient to proceed, 
not when descriptions are maximally precise.

Step 1: Identify the Start of the Current Grounding Episodes
Focus only the recent history of the current conversation as grounding episodes, 
where an episode targets:
Piece grounding (Which piece?)
Location grounding (Where should it go?)
Both (one utterance may initiate two cycles)

starting with a presentation of a referent, a piece, a location or both.

Step 2: Analyze Each Grounding Cycle
For the current episode, produce an analyses on Pieces and Locations and 
annotate the current turn for the types "Presentation", "Clarification", "Correction", "Acceptance".

Presentation:
A presentation is any utterance that proposes a new candidate meaning (piece or location or both) with 
no reference to previous referents for the addressee to understand and potentially accept. 
This is the start of the grounding cycle.
Examples:
"Place the striped one, ..." 
"Put the blue square left to this."

Clarification:
A request for more information because the current presentation is underspecified or ambiguous.
Examples:
"Which one?"
"Down from where?"
"Do you mean the left or the right one?"

Correction:
A turn that rejects an incorrect interpretation and replaces it with the correct one.
Examples:
"No, not that one."
"Not below the blue below the red."

Repair:
Additional descriptive material added to fix a problem without explicitly rejecting the 
prior reference. This can be self-repair
Examples:
"The striped one next to the square."
"Two down starting from the top row."
"Move it up"

Acceptance:
Acceptance is evidence that both participants treat the reference as successfully understood.
Acceptance completes the grounding cycle.
Examples:
Implicit (correct piece selected, conversation continues e.g., new presentation)
Explicit ("Okay," "Got it")

Followup:
Moves the interaction to the next step or puzzle piece. Assumes grounding is complete.
Adds no semantic content to the current grounding target. Not a presentation. Not a repair. 
Not acceptance (though it may follow acceptance)
Examples:
“What should I do next?”
“Okay, next piece.”
“Let’s move on.”
"What is next?"

Step 3 Return your annotations
Return your annotations as list of:
grounding_phrase: Substring of the current utterance acting performing the grounding
grounding_type: Grounding type
referent_phrase: Phrase presenting the referent prior in the conversation (Piece and Location)
referent: Piece or Location or Both or None

OUTPUT FORMAT (STRICT)
For each grounding expression, output a row in JSON array format:

```json
{
    "annotations": [{   
      "grounding_phrase": "<exact text>",  
      "grounding_type": "<Presentation|Clarification|Correction|Repair|Acceptance|Followup>", 
      "referent_phrase": "<exact text|None>",  
      "referent": "<Location|Piece|Both|None>"
      "justification": "<1–2 sentences citing observable evidence>"  
    },
    ...] 
}
```

Input

Turn to annotate:
<TURN>  
{turn}
</TURN>  

Context (optional):

<TURN_CONTEXT>  
{conversation_before} 
</TURN_CONTEXT>  
"""

prompt5 = """
Compare the two puzzles in this image. Does the target puzzle match the final results?

Decide:
exact match: the pieces and locations match excatly
partial match 1: pieces correct, but some locations are wrong
partial match 2: pieces are not correct, but location are corrects
no match: the pieces and location do not match

Return your evaluation as JSON object:
```json
{
 "result": "exact match / partial match 1 / partial match 2 / no match"
}
```
"""
prompt6 = """
You are an expert discourse annotation judge trained in:
Conversational grounding 
Referential communication
Human–Computer Interaction and collaborative task analysis

Your task is to annotate referring expressions in a conversation used during a collaborative task.
You must be conservative, consistent, and theory‑driven.
Do not infer intent beyond observable evidence in the transcript.

There are two roles: The helper does not see all available puzzle pieces but sees the target
pattern with the correct locations and pieces. The worker does not see the target puzzle but sees
all available pieces.

{view}

INPUT
You will receive:
A current turn you need to annotate.
A conversation history transcript between two participants engaged in a collaborative task.
The next turn in the conversation after the current turn.
Actions (e.g., object movement) may be described in brackets.

REFERENCE TYPE DEFINITIONS (MANDATORY)
Use only the following codes:

New Label
A stable, previously grounded identifier.
Examples: “ID 4”, “location 1,2”, “the piece at 1,4”
Assumed unique due to prior presentation

Definite Description
A referring expression that is treated as uniquely identifying by the addressee, as evidenced by 
acceptance without repair.
Examples: “the one with two blue stripes”, “the piece on the left”

Ambiguous Description 
A referring expression that fails to uniquely identify a referent for the addressee, as evidenced 
by clarification, hesitation, or repair.
Examples: “the blue one” (multiple candidates), followed by "which blue piece do you mean?"

Demonstrative Pronoun
A reduced referring expression that points to a previously established discourse entity, 
including events or actions.
Examples: “that”, “this one”

Personal Pronoun
A reduced referring expression that points to a previously established discourse entity, 
including pieces and locations, events or actions.
Examples: “it”, “them”

Elliptic (Zero Reference)
Referent is omitted entirely
Meaning recovered from context or action
Examples: “okay, now left”, silent action after instruction

REFERENT DEFINITIONS (MANDATORY)
Use only the following codes:

Location
The phrase is used to refer to a location on the board.
Example: "Place the green piece right to that" (two references: possible ambiguous description of a piece and demonstrative for location)

Piece
The phrase is used to refer to a puzzle piece.
Example: "Move it on down"

Action
The phrase is used to refer to a performed action.
Example: "Was that correct?"

DECISION RULES (IMPORTANT)

Ambiguity is interactional
- If clarification occurs in the next turn, the reference is Ambiguous Description, even if 
linguistically definite.

Pronouns require an antecedent
- If no clear prior referent exists, classify as Ambiguous Description, not Pronouns.

Ellipsis requires strong grounding
- If meaning relies on prior shared context or visible action, use ELL.

Do not upgrade references
- A description only becomes PN after repeated successful grounding.

Use observable evidence only
- Do not assume understanding unless acceptance is shown.

OUTPUT FORMAT (STRICT)
For each referring expression, output a row in JSON array format:

```json
{
    "annotations": [{   
      "referring_phrase": "<exact text>",  
      "reference_type": "<Definite Description|Ambiguous Description|Demonstrative Pronoun|Personal Pronoun|Ellipsis>", 
      "referent_phrase": "<exact text|None>",  
      "referent": "<Location|Piece|Action|None>"
      "justification": "<1–2 sentences citing observable evidence>"  
    },
    ...] 
}
```
If an utterance contains multiple referring expressions, annotate each separately.
If no referring expression is present, omit the utterance.

EXAMPLE (FOR CALIBRATION ONLY)
Input:

Turn to annotate:
<TURN>  
A: Put the blue one next to it.
</TURN>  

Context (optional):
<TURN_CONTEXT>  
A: Put the red piece to the middle of the board.
B: Ok. What is next?
</TURN_CONTEXT>  

<NEXT_TURN>  
B: Which blue piece?.
</NEXT_TURN>  

Return:
{
    "annotations": [{  
      "referring_phrase": "the blue one",  
      "reference_type": "Ambiguous Description", 
      "referent_phrase": "None",  
      "referent": "Piece"
      "justification": "Multiple blue objects exist, and the addressee requests clarification."  
    },
    {  
      "referring_phrase": "it",  
      "reference_type": "Personal Pronoun", 
      "referent_phrase": "the red piece",  
      "referent": "Piece"
      "justification": "Multiple blue objects exist, and the addressee requests clarification."  
    }]  
}

FINAL INSTRUCTIONS
Be consistent across the entire transcript
Prefer simpler categories when uncertain
Justifications must reference dialogue evidence, not assumptions
Output only valid JSON, no commentary

Input

Current turn to annotate:
<TURN>  
{turn}
</TURN>  

Context (optional):
<TURN_CONTEXT>  
{conversation_before} 
</TURN_CONTEXT>  

Next turn (optional)
<NEXY_TURN>  
{next_turn}
</NEXT_TURN> 
"""

In [ ]:
import json
with open("sessions_12_08_2025.json", encoding="utf-8") as f:
    sessions = json.load(f)
    
results = []

for session, data in sessions.items():
    puzzles = data["puzzles"]
    
    for i, puzzle in puzzles.items():
        if puzzle["puzzle_details"]["config"]["puzzle_group"] == "Shared view":
            view = "The helper can see the current progress of the worker including the their placed pieces."
        else:
            view = "The helper cannot see the current progress of the worker and is not aware of where they places what pieces."
        
        annotations = []
        messages = [m for m in puzzle["messages"] if m["sender"] != None]
        conv = ""
        
        for j, m in enumerate(messages): 
            if m["sender"] == None:
                continue
                
            turn = m["sender"] + ": " + m["message"]
            if j < len(messages) - 1:
                next_turn = messages[j+1]["sender"] + ": " + messages[j+1]["message"]
            else:
                next_turn = ""

            prompt = prompt4.replace("{conversation_before}", conv).replace("{turn}", turn).replace("{next_turn}", next_turn).replace("{view}", view) + "\nAdd explanations"
            
            annotation_results = gpt_request(prompt, model="gpt-5.2-chat_2025-12-11", api_version="2024-12-01-preview")
            json_anno = get_json(annotation_results)
            if len(json_anno) > 0:
                annotations.append(json_anno)
            
            conv = conv + m["sender"] + ": " + m["message"] + "\n"
            # break
    #         if len(annotations) > 7:
    #             break
        # break
        puzzle["messages"] = messages
        puzzle["annotations"] = annotations

        
    # break
    results.append({session: data})
    with open("sessions_12_08_2025_annotations_new_grounding_acts2.json", "w", encoding="utf-8") as f:
        json.dump(results, f)
    # break

# print(json.dumps(conv[1], indent=3))
# conv

In [ ]:
# with open("sessions_12_08_2025_annotations_new_refs_only_3.json", encoding="utf-8") as f:
#     tmp = json.load(f)
#     sessions_ref = {}
#     for t in tmp:
#         for k, v in t.items():
#             sessions_ref[k] =  v
with open("sessions_12_08_2025_annotations_new_grounding_acts2.json", encoding="utf-8") as f:
    tmp = json.load(f)
    sessions_ground = {}
    for t in tmp:
        for k, v in t.items():
            sessions_ground[k] =  v

In [ ]:
import pandas as pd  
import matplotlib.pyplot as plt  
import numpy as np  
  
references = [
"Definite Description",
"Demonstrative Pronoun",
"Personal Pronoun",
"Ambiguous Description",
"New Label",
"Ellipsis"
# "ambiguous"
]

groundings = [
    "Acceptance",
    "Presentation",
    "Clarification",
    "Correction",
    "Repair",
    "Followup"
]

def compare_senders_labels_barchart(sessions, category, worker="Human"):  
  
    rows = []  
    for session, data in sessions.items():  
        if worker == "Human":  
            helper = "AI"  
            if session not in human_worker:  
                continue  
        else:  
            helper = "Human"  
            if session not in human_helper:  
                continue  
  
        puzzles = data["puzzles"]  
        for pid, data in puzzles.items():  
            if pid in ["6", "7", "8", "9"]:
                continue
            for msg, anns in zip(data["messages"], data["annotations"]):  
                # print(anns)
                if isinstance(anns, str):
                    continue
                for ann in anns["annotations"]:
                    label_val = ann[category]
                    if ann["referent_phrase"] == "None":
                        continue
                    
                    if category == "grounding_type" and not label_val in groundings:
                        continue
                    if category == "reference_type" and not label_val in references:
                        continue
                    # if category == "vocabulary_alignment" and not label_val in vocab:
                    #     continue
                    if label_val is None or label_val == "none" or label_val == "unverifiable" or label_val == "establish_reference" or label_val == "situation_awareness":  
                        continue   # skip None labels  
                    if label_val == "Followup":
                        label_val = "Acceptance"
                    rows.append({  
                        "puzzle": pid,  
                        "sender": msg["sender"],  
                        "label": label_val  
                    })  
  
    df = pd.DataFrame(rows)  
  
    counts = df.groupby(["puzzle", "sender", "label"]).size().unstack(fill_value=0)  
    totals = df.groupby(["puzzle", "sender"]).size()  
  
    puzzles_sorted = sorted(df["puzzle"].unique())  
    labels = list(counts.columns)  
  
    x = np.arange(len(puzzles_sorted))  
    width = 0.8 / len(labels)  
  
    fig, (ax_helper, ax_worker) = plt.subplots(1, 2, figsize=(16,5), sharey=True)  
  
    for i, label in enumerate(labels):  
        helper_vals = []  
        worker_vals = []  
  
        for p in puzzles_sorted:  
            if (p, "helper") in totals.index:  
                helper_vals.append(  
                    counts.loc[(p, "helper")].get(label, 0) /  
                    totals.loc[(p, "helper")]  
                )  
            else:  
                helper_vals.append(0)  
  
            if (p, "worker") in totals.index:  
                worker_vals.append(  
                    counts.loc[(p, "worker")].get(label, 0) /  
                    totals.loc[(p, "worker")]  
                )  
            else:  
                worker_vals.append(0)  
  
        # ----- helper subplot -----  
        ax_helper.plot(  
            x,# + i*width - width*len(labels)/2 - 0.01,  
            helper_vals,  
            # width=width,  
            marker="o",
            label=f"{label}"  
        )  
  
        # ----- worker subplot -----  
        ax_worker.plot(  
            x,# + i*width + width*len(labels)/2 + 0.01,  
            worker_vals,  
            # width=width,  
            marker="o",
            label=f"{label}"  
        )  
  
    for ax in (ax_helper, ax_worker):  
        ax.set_xticks(x)  
        ax.set_xticklabels(puzzles_sorted)  
        ax.set_xlabel("Puzzle ID")  
        ax.set_ylim(0, 1)
  
    ax_helper.set_title(f"{helper} Helper")  
    ax_worker.set_title(f"{worker} Worker")  
  
    ax_helper.set_ylabel("Proportion")  
    ax_worker.legend(bbox_to_anchor=(1.05,1), loc="upper left")  
  
    plt.suptitle(f"{category}: Shared view first with {helper} helper and {worker} worker.") 
    plt.tight_layout()  
    plt.savefig(f"{category}_annotations_{worker}_worker_{helper}_helper_shared_view_first.png", dpi=300, bbox_inches="tight")
    plt.show()  
  
  
# examples  
# compare_senders_labels_barchart(sessions_ref, "reference_type", worker="Human", scope="worker")  
# compare_senders_labels_barchart(sessions_ref, "reference_type", worker="AI")  
# compare_senders_labels_barchart(sessions, "vocabulary_alignment", worker="Human")  
# compare_senders_labels_barchart(sessions, "vocabulary_alignment", worker="AI")  

compare_senders_labels_barchart(sessions_ground, "grounding_type", worker="Human")  
compare_senders_labels_barchart(sessions_ground, "grounding_type", worker="AI")  

In [ ]:
with open("sessions_12_08_2025_annotations_new2.json", encoding="utf-8") as f:
    tmp = json.load(f)
    sessions_ground = {}
    for t in tmp:
        for k, v in t.items():
            sessions_ground[k] =  v

In [ ]:
import pandas as pd  
import matplotlib.pyplot as plt  
import numpy as np  
  
grounding = ["clarify","correct","accept"]   
vocab = ["adopts_partner","introduces_new"]  

def compare_senders_labels_grounding_chart(sessions, worker="Human"):  
    rows = []  
    for session, data in sessions.items():  
        if worker == "Human":  
            helper = "AI"  
            if session not in human_worker:  
                continue  
        else:  
            helper = "Human"  
            if session not in human_helper:  
                continue  
  
        puzzles = data["puzzles"]  
        for pid, data in puzzles.items():  
            if pid in ["6", "7", "8", "9"]:
                continue
            for t, (msg, anns) in enumerate(zip(data["messages"], data["annotations"])):  
                # print(anns)
                grounding_act = anns["grounding_act"]
                if grounding_act not in grounding:
                    continue
                vocabulary_alignment = anns["vocabulary_alignment"]
                # if category == "grounding_act" and label_val in grounding:
                #     continue

                # if category == "vocabulary_alignment" and not label_val in vocab:
                #     continue

                rows.append({  
                    "puzzle": pid,  
                    "turn": t,
                    "sender": msg["sender"],  
                    "label": grounding_act,
                    # "vocabulary_alignment": vocabulary_alignment
                })  
  
    df = pd.DataFrame(rows)  
    # print(df)
    # return
  
    counts = df.groupby(["puzzle", "sender", "label"]).size().unstack(fill_value=0)  
    totals = df.groupby(["puzzle", "sender"]).size()  
  
    puzzles_sorted = sorted(df["puzzle"].unique())  
    labels = list(counts.columns)  
  
    x = np.arange(len(puzzles_sorted))  
    width = 0.8 / len(labels)  
  
    fig, (ax_helper, ax_worker) = plt.subplots(1, 2, figsize=(16,5), sharey=True)  
  
    for i, label in enumerate(labels):  
        helper_vals = []  
        worker_vals = []  
  
        for p in puzzles_sorted:  
            if (p, "helper") in totals.index:  
                helper_vals.append(  
                    counts.loc[(p, "helper")].get(label, 0) /  
                    totals.loc[(p, "helper")]  
                )  
            else:  
                helper_vals.append(0)  
  
            if (p, "worker") in totals.index:  
                worker_vals.append(  
                    counts.loc[(p, "worker")].get(label, 0) /  
                    totals.loc[(p, "worker")]  
                )  
            else:  
                worker_vals.append(0)  
  
        # ----- helper subplot -----  
        ax_helper.plot(  
            x,# + i*width - width*len(labels)/2 - 0.01,  
            helper_vals,  
            # width=width,  
            marker="o",
            label=f"{label}"  
        )  
  
        # ----- worker subplot -----  
        ax_worker.plot(  
            x,# + i*width + width*len(labels)/2 + 0.01,  
            worker_vals,  
            # width=width,  
            marker="o",
            label=f"{label}"  
        )  
  
    for ax in (ax_helper, ax_worker):  
        ax.set_xticks(x)  
        ax.set_xticklabels(puzzles_sorted)  
        ax.set_xlabel("Puzzle ID")  
  
    ax_helper.set_title(f"{helper} Helper")  
    ax_worker.set_title(f"{worker} Worker")  
  
    ax_helper.set_ylabel("Proportion")  
    ax_worker.legend(bbox_to_anchor=(1.05,1), loc="upper left")  
  
    plt.suptitle(f"{category}: Shared view first with {helper} helper and {worker} worker.") 
    plt.tight_layout()  
    plt.savefig(f"grounding_annotations_{worker}_worker_{helper}_helper_shared_view_first.png", dpi=300, bbox_inches="tight")
    plt.show()  
  
  
# examples  
compare_senders_labels_grounding_chart(sessions_ground, worker="Human") 

In [ ]:
# find definite references (descriptions) and their referent
# for each previous turn check if referent is containted

rows = []
worker = "Human"
for session, data in sessions_ref.items():  
    # if worker == "Human":  
    #     helper = "AI"  
    #     if session not in human_worker:  
    #         continue  
    # else:  
    #     helper = "Human"  
    #     if session not in human_helper:  
    #         continue  

    puzzles = data["puzzles"]  
    messages = []
    referents = set()
    for pid, data in puzzles.items():  
        if pid in ["6", "7", "8", "9"]:
            continue
        messages = []
        pids = []
        referents_from_AI = set()
        
        referents_from_human = set()

        for i, (msg, anns) in enumerate(zip(data["messages"], data["annotations"])):  
            # print(anns)
            
            messages.append(msg)
            pids.append(pid)
            if isinstance(anns, str):
                continue
            
            for ann in anns["annotations"]:
                # label_val = ann[category]
                # if ann["referent_phrase"] == "None" or "Description" not in ann["reference_type"]:
                #     continue
                if "Description" not in ann["reference_type"]:
                    continue
                else:
                    referent = ann["referring_phrase"].lower()
                    
                    if session in human_worker:
                        if msg["sender"] == "worker":
                            
                            referents_from_human.add(referent)
                            
                        else:
                            
                            referents.add(referent)
                            referents_from_AI.add(referent)
                    elif session in human_helper:
                        if msg["sender"] == "helper":
                            # if referent == "the pink piece with four wavy lines":
                            #     print(ann, msg, session)
                            referents_from_human.add(referent)
                            
                        else:
                            referents.add(referent)
                            referents_from_AI.add(referent)

                    

        rows.append({  
            "puzzle": pid, 
            "union_vocab_size": len(referents_from_human | referents_from_AI),
            "joint_vocab_size": len(referents_from_human & referents_from_AI),
            "AI_vocab_size": len(referents_from_AI),
            "human_vocab_size": len(referents_from_human)
        })  

    # break

df = pd.DataFrame(rows) 
# print(df)
import matplotlib.pyplot as plt    
  
grouped = (  
    df.groupby('puzzle')[['union_vocab_size','joint_vocab_size', 'AI_vocab_size', 'human_vocab_size']]  
      .mean()  
)   
plt.figure(figsize=(8, 5))  
# plt.plot(grouped.index, grouped['union_vocab_size'], marker='o', label='Union')    
plt.plot(grouped.index, grouped['joint_vocab_size'], marker='o', label='Joint')  
plt.plot(grouped.index, grouped['AI_vocab_size'], marker='o', label='AI')  
plt.plot(grouped.index, grouped['human_vocab_size'], marker='o', label='Human')  
  
plt.xlabel('Puzzle')  
plt.ylabel('Vocabulary Size')  
plt.title('Vocabulary Size by Puzzle')  
plt.legend()  
plt.grid(True)  
  
plt.show()  

In [ ]:
path = "visualization_output/comparison_images"
import os  
import json
annotations = []
for f in os.listdir(path):  
    if f.lower().endswith(".png"):  
        print(f)  
        annotation_results = gpt_request_text_and_image_upload(prompt5, path+"/"+f, model="gpt-5-chat_2025-08-07", api_version="2024-12-01-preview")
        json_anno = get_json(annotation_results)
        
        if len(json_anno) > 0:
            annotations.append({f: json_anno["result"]})
        else:
            annotations.append({f: annotation_results})
        
        with open("puzzle_task_performance.json", "w", encoding="utf-8") as f:
            json.dump(annotations, f)
# puzzle["annotation"]

In [ ]:
import json
with open("puzzle_task_performance.json", encoding="utf-8") as f:
    task_completions = json.load(f)

In [ ]:
from collections import defaultdict
session_to_results = defaultdict(list)
for task in task_completions:
    for k, v in task.items():
        session_to_results[k[9:16]].append(v)

In [ ]:
from collections import Counter
puzzle_to_summary = defaultdict(Counter)
for k, v in session_to_results.items():
    for i, r in enumerate(v):
        if r.endswith("1") or r.endswith("2"):
            r = r[:-2]
        puzzle_to_summary[i].update([r])

In [ ]:
# puzzle_to_summary

In [ ]:
import pandas as pd  
import matplotlib.pyplot as plt   
  
df = pd.DataFrame(puzzle_to_summary).T  
  
custom_colors = ['orange', 'green', 'red']
df[["no match", "partial match", "exact match"]].plot(  
    kind="bar",  
    stacked=True,  
    color=["red", "orange", "green"]  
) 
  
plt.xticks(rotation=0)  
plt.xlabel("Puzzle index")  
plt.ylabel("Count")  
plt.title("Puzzle match results")  
  
plt.legend(title="Match type", bbox_to_anchor=(1.05, 1), loc="upper left")  
  
plt.tight_layout()  
plt.savefig(f"task_performances_shared_view_first.png", dpi=300, bbox_inches="tight")  
plt.show()   


In [ ]:
# puzzle_to_summary
l1 = []
l2 = []
# session_to_results
rows = []
for session, performances in session_to_results.items():
    # v = np.mean([0 if v == "no match" else 1 for v in  performances[1:5]])
    # l1.append(v)
    # v = np.mean([0 if v == "no match" else 1 for v in  performances[5:9]])
    # l2.append(v)
    
    for i in [1,2,3,4]:
        v = 1
        if performances[i] == "no match":
            v = 0
        rows.append({"session": session, "puzzle": i, "success": v})
        l1.append(v)
    for i in [5,6,7,8]:
        v = 1
        if performances[i] == "no match":
            v = 0
        l2.append(v)

print(l1, l2)

In [ ]:
# !pip install statsmodels  
import pandas as pd
import statsmodels.api as sm  
import statsmodels.formula.api as smf  
df_success = pd.DataFrame.from_dict(rows) 
  
# Keep only puzzles 2–5  

df_succ = df_success[df_success["puzzle"].isin([1, 2, 3, 4])].copy()  
df_succ["puzzle"] = df_succ["puzzle"].astype("category")  
  
gee = smf.gee(  
    "success ~ C(puzzle)",  
    groups="session",  
    data=df_succ,  
    family=sm.families.Binomial()  
)  
  
result = gee.fit()  
print(result.summary()) 

In [ ]:
from scipy.stats import spearmanr, chi2_contingency, fisher_exact
from statsmodels.stats.contingency_tables import mcnemar 
import numpy as np
a = np.array(l1)
b = np.array(l2)
table = [
    [np.sum(a == 1), np.sum(a == 0)],
    [np.sum(b == 1), np.sum(b == 0)]
]
# _, p = fisher_exact(table)
res = mcnemar(table, exact=True)
print(res)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar  
import numpy as np  
  
a = np.array(l1)  
b = np.array(l2)  
  
table = [  
    [np.sum((a == 1) & (b == 1)),  # A=1, B=0  
     np.sum((a == 1) & (b == 0))],  
    [np.sum((a == 0) & (b == 1)),  # A=0, B=1  
     np.sum((a == 0) & (b == 0))]  
]  
  
res = mcnemar(table, exact=True)  
print(res)  

In [ ]:
# =========================  
# LLM JUDGE: STATEFUL COMMON GROUND TRACKER  
# ACROSS TURNS AND ACROSS PUZZLES  
# =========================  
  
from typing import Dict, Any, List  
import json  
from textwrap import dedent  
  
  
# ---------- Prompt Template ----------  

mention_detection_prompt = """
You are supposed to extract entities from an utterance from a conversation
between a helper and a worker to solve a puzzle together. They have
to coordinate such that the helper instruct the worker to place puzzle
pieces step by step to form a target patterns on a board. 

Extract entities mentions of puzzle pieces and locations from this utterance used 
to perform the task of solving the puzzle. Entity mentions can also be references, 
or pronouns of the pieces or locations. Return the list of all mentions.

Utterance
{turn}

Return the list of terms as JSON object:
```json
{
 "mentions": [list of all mentions]
}
```
"""
  
def process_puzzles(  
    puzzles: Dict[str, Dict[str, Any]],  
) -> Dict[str, Any]:  
    """  
    puzzles = {  
      "puzzle_1": {"messages": [turn_dicts...]},  
      "puzzle_2": {"messages": [...]}  
    }  
    """  
  
    prior_context = ""  
    puzzle_results = {}  
    errors = []
  
    for puzzle_id, puzzle_data in puzzles.items():  
        turn_analyses = []  
  
        for turn in puzzle_data["messages"]:  
  
            prompt = mention_detection_prompt.replace("{turn}", turn['message'])

            annotation_results = gpt_request(prompt, model="gpt-5-chat_2025-08-07", api_version="2024-12-01-preview")
            annotation_results = annotation_results.replace("null", "None")
            response = get_json(annotation_results)
            try:
                turn_analyses.append(response)  
            except:
                turn_analyses.append(None)  
                errors.append(annotation_results)
                # return annotation_results
  
        puzzle_results[puzzle_id] = {  
            "entities": turn_analyses  
        }  
  
    return puzzle_results

# res = process_puzzles(sessions['5c68952d-1810-4cf6-a1f0-bc28c020f7eb']['puzzles'])

In [ ]:
for sess, v in sessions.items():
    res = process_puzzles(v["puzzles"])
    v["entities"] = res
# sessions['5c68952d-1810-4cf6-a1f0-bc28c020f7eb']['puzzles']
# print(res)
    with open("sessions_with_mentions_2.json", "w", encoding="utf-8") as f:
        json.dump(sessions, f)
# with open("test_vocab.json", encoding="utf-8") as f:
#     res =json.load(f)

In [ ]:
# print(json.dumps(res, indent=3))